# A5 — Phân tích đã khóa và Explainable AI

Phân tích dự đoán từ A4 trên **tập test đã khóa**. A5 không huấn luyện lại hay lựa chọn mô hình từ test.

V3 implementation and command templates only. Primary SelectFromModel is implemented: weighted B3-configured XGBoost gain, exact k=21/15/10/5, stable original-column ties, per-fold fits and full-primary-training refits. Reserved validation selects k against locked full-feature A2 AP within 0.01; selected A3 is separately reported and A2 remains fallback. CV diagnostics are conditional on B3 settings selected on the same folds, not nested CV. ANOVA is separately saved QA only. Official training/evaluation authorization remains PENDING. F2 retains the ascending 199-point grid and smallest exact-tie threshold. All three paired AP comparisons use one locked V3 run. Historical B LightGBM scores are not official XGBoost evidence. SHAP defaults are 500/100; LIME remains 1000 perturbations.
Real **TRAINING-ONLY GroupSplit AP sensitivity** is complete: Stratified mean AP 0.4362740784; Grouped 0.4358097385; Group minus Stratified -0.0004643398; zero grouped-fold exact-X overlap. These descriptive training-CV results do not select the primary model or establish held-out performance. Official full-data A4/A5 and actual SHAP/LIME runtime remain pending. Verified numeric mapping is three-class 0/1/2 to binary 0/0/1; no labels changed.


## 1. Chọn nguồn A4 đã chạy trên CSV thật

Use a reviewed **full repository copy**, including root `src/diabetes_research/`, component V3 config/amendment, scripts and both notebooks, `shared_splits/` archive/manifest/verifier, and root packaging/dependency metadata. Component-only mounting is insufficient. In a local clean copy, open from its root, component or notebooks directory. On Kaggle, copy the full repository to `/kaggle/working/UET.CS2046_7_Nhom5` (or another writable path), then set the notebook working directory to that root before the first code cell; discovery searches cwd and parents. Mount the original CSV separately and edit `DATA` in that cell to its mount path. Keep output paths writable and install editable from the repository root. These are reviewed templates; actual Kaggle reproduction and official A4/A5 remain pending.


In [ ]:
from pathlib import Path
# Discover a full reviewed repository copy from cwd and parents.
_required = (
    'src/diabetes_research/modeling.py',
    'src/diabetes_research/sensitivity_contract.py',
    'research/lead_modeling/a4_a5/config/A2_protocol_v3.yaml',
    'research/lead_modeling/a4_a5/config/PROTOCOL_AMENDMENTS_V3.json',
    'research/lead_modeling/a4_a5/scripts/run_a4.py',
    'research/lead_modeling/a4_a5/scripts/run_a5.py',
    'research/lead_modeling/a4_a5/scripts/run_group_sensitivity.py',
    'research/lead_modeling/a4_a5/notebooks/A4_Training_Protocol_v2.ipynb',
    'research/lead_modeling/a4_a5/notebooks/A5_Evaluation_XAI_v2.ipynb',
    'research/lead_modeling/a4_a5/requirements.txt',
    'shared_splits/shared_split_indices.npz',
    'shared_splits/shared_split_manifest.json',
    'shared_splits/verify_shared_split.py',
    'pyproject.toml',
    'requirements.txt',
)
_here = Path.cwd().resolve()
REPO = None
for _candidate in (_here, *_here.parents):
    if all((_candidate / _relative).is_file() for _relative in _required):
        REPO = _candidate
        break
if REPO is None:
    raise FileNotFoundError('Use a full V3 repository copy (root src, component config/scripts/notebooks, shared_splits and packaging); set cwd inside that copy before this cell')
ROOT = REPO / 'research' / 'lead_modeling' / 'a4_a5'
DATA=ROOT/'data'/'diabetes_binary_health_indicators_BRFSS2015.csv'
A4=ROOT/'results'/'a4_v3_001'
OUT=ROOT/'results'/'a5_v3_001'
assert (A4/'manifest.json').is_file()
import json
manifest=json.loads((A4/'manifest.json').read_text())
assert manifest['status']=='RESEARCH_TEST_COMPLETE', 'Không dùng smoke/demo làm kết quả'
manifest['status']
assert manifest['manifest_version']==3, 'Require a fresh V3 A4 run'


## 2. Phân tích thống kê + bootstrap 1.000 lần

In [ ]:
import subprocess,sys
cmd=[sys.executable,str(ROOT/'scripts'/'run_a5.py'),
     '--data',str(DATA),'--a4-dir',str(A4),'--output-dir',str(OUT),'--bootstraps','1000']
subprocess.run(cmd,check=True)

## 3. Kiểm tra bảng số liệu và đồ thị

Tất cả là kết quả từ code sau khi thực sự chạy với CSV Kaggle.

In [ ]:
import pandas as pd
from IPython.display import display,Image
display(pd.read_csv(A4/'test_metrics.csv'))
print(json.dumps(json.loads((OUT/'artifact_verification.json').read_text()),indent=2))
print(json.dumps(json.loads((OUT/'paired_bootstrap_summary.json').read_text()),indent=2,ensure_ascii=False))
display(Image(filename=str(OUT/'precision_recall_test.png')))
display(Image(filename=str(OUT/'confusion_matrix_final.png')))

## 4. Explainability (tùy chọn)

Cài `requirements-explain.txt` nếu cần SHAP/LIME. Sử dụng thư mục kết quả khác. Chỉ diễn giải hành vi mô hình, không suy ra nguyên nhân y tế.

In [ ]:
# Bỏ comment khi đã cài shap/lime và muốn phân tích thêm:
# XAI_OUT=ROOT/'results'/'a5_v3_with_xai_001'
# cmd=[sys.executable,str(ROOT/'scripts'/'run_a5.py'),
#      '--data',str(DATA),'--a4-dir',str(A4),'--output-dir',str(XAI_OUT),
#      '--bootstraps','1000','--shap','--lime',
#      '--shap-samples','500','--shap-background-samples','100']
# subprocess.run(cmd,check=True)

## 5. Viết Discussion

Dùng `docs/A5_Results_Discussion_Template.md` và file Word A5 để điền số liệu thật, nêu cả kết quả âm/giới hạn. Ghi lại các tương tác AI và cách kiểm chứng.